In [ ]:
from pathlib import Path

all_words = set()      # unique vocabulary
file_words = {}        # file → words mapping


def read_align_files(directory_path):
    p = Path(directory_path)

    if not p.is_dir():
        print(f"Error: {directory_path} is not a valid directory.")
        return
    count=0
    for file_path in p.iterdir():
        count+=1
        if file_path.is_file() and file_path.suffix == ".align":

            words_in_file = []

            try:
                with open(file_path, "r", encoding="utf-8") as f:

                    for line in f:
                        parts = line.strip().split()

                        if len(parts) == 3:
                            start, end, word = parts

                            # skip silence tokens if you want
                            if word != "sil":
                                words_in_file.append(word)
                                all_words.add(word)

                file_words[file_path.name] = words_in_file

            except Exception as e:
                print(f"Error reading {file_path.name}: {e}")
    print(count)


# Run
read_align_files("./s1")

# Convert set → list
all_words = list(all_words)


print("All Words Vocabulary:")
print(all_words)

print("\nWords per File:")
print(file_words)

1000
All Words Vocabulary:
['blue', 'd', 's', 'lay', 'j', 'soon', 'a', 'by', 'white', 'please', 'z', 't', 'sp', 'q', 'set', 'eight', 'f', 'green', 'again', 'nine', 'l', 'i', 'x', 'zero', 'seven', 'two', 'o', 'p', 'with', 'b', 'k', 'bin', 'in', 'five', 'now', 'red', 'r', 'six', 'n', 'v', 'y', 'm', 'at', 'one', 'h', 'c', 'place', 'e', 'four', 'three', 'g', 'u']

Words per File:
{'bbaf2n.align': ['bin', 'blue', 'at', 'f', 'two', 'now'], 'bbaf3s.align': ['bin', 'blue', 'at', 'f', 'three', 'soon'], 'bbaf4p.align': ['bin', 'blue', 'at', 'f', 'four', 'please'], 'bbaf5a.align': ['bin', 'blue', 'at', 'f', 'five', 'again'], 'bbal6n.align': ['bin', 'blue', 'at', 'l', 'six', 'now'], 'bbal7s.align': ['bin', 'blue', 'at', 'l', 'seven', 'soon'], 'bbal8p.align': ['bin', 'blue', 'at', 'l', 'eight', 'please'], 'bbal9a.align': ['bin', 'blue', 'at', 'l', 'nine', 'again'], 'bbas1s.align': ['bin', 'blue', 'at', 's', 'one', 'soon'], 'bbas2p.align': ['bin', 'blue', 'at', 's', 'two', 'please'], 'bbas3a.align':

In [11]:
from pathlib import Path

IGNORE = {"sil", "sp"}   # remove pause tokens

all_words = set()

def collect_words(directory):
    p = Path(directory)

    for file_path in p.glob("*.align"):

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:

                parts = line.strip().split()

                if len(parts) == 3:
                    _, _, word = parts

                    all_words.add(word)


collect_words("./s1")

# Convert to sorted list for clean printing
final_words = sorted(all_words)
print(len(final_words))
print("Final Vocabulary:")
print(" ".join(final_words))

53
Final Vocabulary:
a again at b bin blue by c d e eight f five four g green h i in j k l lay m n nine now o one p place please q r red s set seven sil six soon sp t three two u v white with x y z zero


In [12]:
VISeme_MAP = {
    "round": {"blue", "two", "u", "o", "zero"},
    "wide": {"three", "e", "green", "please"},
    "closed": {"bin", "b", "p", "m"},
    "sibilant": {"six", "seven", "s", "z"},
    "neutral": {"at", "with", "again", "now", "place", "set", "lay", "white", "red"}
}

In [14]:
from pathlib import Path
import subprocess

ALIGN_DIR = Path(r"C:\Users\adity\archive\processed\s1")
VIDEO_DIR = Path(r"C:\Users\adity\archive\processed\video")
OUTPUT_DIR = Path(r"C:\Users\adity\archive\viseme_dataset")

IGNORE = {"sil", "sp"}

VISeme_MAP = {
    "viseme_round": {"blue", "two", "u", "o", "zero"},
    "viseme_wide": {"three", "e", "green", "please"},
    "viseme_closed": {"bin", "b", "p", "m"},
    "viseme_sibilant": {"six", "seven", "s", "z"},
    "viseme_neutral": {"at", "with", "again", "now", "place", "set", "lay", "white", "red"}
}


def get_viseme(word):
    for viseme, words in VISeme_MAP.items():
        if word in words:
            return viseme
    return "viseme_neutral"


def extract_clip(video_path, start, end, output_path):
    duration = end - start

    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-ss", str(start),
        "-t", str(duration),
        "-c", "copy",
        str(output_path)
    ]

    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def process():
    for align_file in ALIGN_DIR.glob("*.align"):

        video_file = VIDEO_DIR / (align_file.stem + ".mp4")

        if not video_file.exists():
            print("Missing video:", video_file)
            continue

        with open(align_file, "r") as f:
            lines = f.readlines()

        for i, line in enumerate(lines):

            parts = line.strip().split()

            if len(parts) != 3:
                continue

            start, end, word = parts

            if word in IGNORE:
                continue

            # GRID timestamps are in 1/10000 sec
            start = int(start) / 10000
            end = int(end) / 10000

            viseme = get_viseme(word)

            viseme_dir = OUTPUT_DIR / viseme
            viseme_dir.mkdir(parents=True, exist_ok=True)

            output_file = viseme_dir / f"{align_file.stem}_{i}_{word}.mp4"

            extract_clip(video_file, start, end, output_file)

        print("Processed:", align_file.name)


process()

Processed: bbaf2n.align
Processed: bbaf3s.align
Processed: bbaf4p.align
Processed: bbaf5a.align
Processed: bbal6n.align
Processed: bbal7s.align
Processed: bbal8p.align
Processed: bbal9a.align
Processed: bbas1s.align
Processed: bbas2p.align
Processed: bbas3a.align
Processed: bbaszn.align
Processed: bbaz4n.align
Processed: bbaz5s.align
Processed: bbaz6p.align
Processed: bbaz7a.align
Processed: bbbf6n.align
Processed: bbbf7s.align
Processed: bbbf8p.align
Processed: bbbf9a.align
Processed: bbbm1s.align
Processed: bbbm2p.align
Processed: bbbm3a.align
Processed: bbbmzn.align
Processed: bbbs4n.align
Processed: bbbs5s.align
Processed: bbbs6p.align
Processed: bbbs7a.align
Processed: bbbz8n.align
Processed: bbbz9s.align
Processed: bbie8n.align
Processed: bbie9s.align
Processed: bbif1a.align
Processed: bbifzp.align
Processed: bbil2n.align
Processed: bbil3s.align
Processed: bbil4p.align
Processed: bbil5a.align
Processed: bbir6n.align
Processed: bbir7s.align
Processed: bbir8p.align
Processed: bbir9

In [10]:
from pathlib import Path
import subprocess
import cv2
import mediapipe as mp
import numpy as np
from collections import defaultdict
from tqdm import tqdm

# =========================
# PATHS
# =========================

ALIGN_DIR = Path(r"C:\Users\adity\archive\processed\s1")
VIDEO_DIR = Path(r"C:\Users\adity\archive\processed\video")
OUTPUT_DIR = Path(r"C:\Users\adity\archive\processed\viseme_dataset_final")

IGNORE = {"sil", "sp"}
CLIPS_PER_WORD = 3

VOCAB = [
"a","again","at","b","bin","blue","by","c","d","e","eight","f","five","four",
"g","green","h","i","in","j","k","l","lay","m","n","nine","now","o","one","p",
"place","please","q","r","red","s","set","seven","six","soon","t","three",
"two","u","v","white","with","x","y","z","zero"
]

# =========================
# MEDIAPIPE INIT
# =========================

mp_face = mp.solutions.face_mesh

face_mesh = mp_face.FaceMesh(
    static_image_mode=False,
    refine_landmarks=True,
    max_num_faces=1
)

# Mouth landmark indices
UPPER_LIP = 13
LOWER_LIP = 14
LEFT_LIP = 78
RIGHT_LIP = 308

# =========================
# HELPERS
# =========================

def find_video(stem):
    for ext in [".mp4", ".mpg", ".mpeg", ".avi"]:
        p = VIDEO_DIR / (stem + ext)
        if p.exists():
            return p
    return None


def face_landmarks(frame):

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = face_mesh.process(rgb)

    if not res.multi_face_landmarks:
        return None

    return res.multi_face_landmarks[0].landmark


def motion_energy(prev, curr):

    if prev is None:
        return 0

    diff = cv2.absdiff(prev, curr)
    return np.mean(diff)


# =========================
# ADVANCED LIP FEATURES
# =========================

def lip_metrics(lm):

    top = lm[UPPER_LIP]
    bottom = lm[LOWER_LIP]
    left = lm[LEFT_LIP]
    right = lm[RIGHT_LIP]

    vertical = abs(top.y - bottom.y)
    horizontal = abs(left.x - right.x)

    if horizontal == 0:
        horizontal = 1e-6

    mar = vertical / horizontal  # mouth aspect ratio
    width = horizontal
    height = vertical

    return mar, width, height


# =========================
# ADVANCED GEOMETRY AI
# =========================

def classify_viseme(mar, width, motion):

    # FULL CLOSED (B,P,M)
    if mar < 0.12:
        return "viseme_closed"

    # PARTIAL CLOSED (F,V)
    if mar < 0.22:
        return "viseme_partial"

    # ROUND (O,U)
    if width < 0.045 and mar > 0.25:
        return "viseme_round"

    # WIDE (EE)
    if width > 0.075 and mar > 0.30:
        return "viseme_wide"

    # LARGE OPEN (AA)
    if mar > 0.55:
        return "viseme_open_big"

    # SMALL OPEN (AH)
    if mar > 0.35:
        return "viseme_open_small"

    # SIBILANT (S,Z)
    if motion > 0.05:
        return "viseme_sibilant"

    return "viseme_neutral"


# =========================
# WINDOW SEARCH (ADAPTIVE)
# =========================

def best_window(video_path, start, end):

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    if fps <= 1:
        fps = 25

    video_length = total_frames / fps

    cap.set(cv2.CAP_PROP_POS_MSEC, start * 1000)

    scores = []
    times = []
    visemes = []

    prev_gray = None
    t = start

    while t <= end:

        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        lm = face_landmarks(frame)

        if lm is None:
            mar = 0
            width = 0
            motion = 0
            score = 0
            vis = "viseme_neutral"
        else:
            mar, width, _ = lip_metrics(lm)
            motion = motion_energy(prev_gray, gray)

            score = mar + 0.5 * motion
            vis = classify_viseme(mar, width, motion)

        scores.append(score)
        times.append(t)
        visemes.append(vis)

        prev_gray = gray
        t += 1 / fps

    cap.release()

    if len(scores) == 0:
        peak_time = (start + end) / 2
        duration = min(0.4, end - start)
        return peak_time, duration, "viseme_neutral"

    scores = np.array(scores)

    peak_idx = int(np.argmax(scores))
    peak_time = times[peak_idx]
    viseme = visemes[peak_idx]

    # ---------- adaptive window ----------
    threshold = scores[peak_idx] * 0.5

    left = peak_idx
    while left > 0 and scores[left - 1] > threshold:
        left -= 1

    right = peak_idx
    while right < len(scores) - 1 and scores[right + 1] > threshold:
        right += 1

    start_time = times[left]
    end_time = times[right]

    duration = end_time - start_time

    # enforce min/max
    duration = max(0.25, min(duration, 0.6))

    start_time = peak_time - duration / 2

    # clamp boundaries
    start_time = max(0, start_time)

    if start_time + duration > video_length:
        start_time = max(0, video_length - duration)

    return start_time, duration, viseme


# =========================
# SAVE CLIP
# =========================

def save_clip(video, start, duration, output):

    cmd = [
        "ffmpeg",
        "-y",
        "-ss", str(start),
        "-i", str(video),
        "-t", str(duration),
        "-vf", "fps=25",
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-preset", "fast",
        "-crf", "23",
        "-c:a", "aac",
        str(output)
    ]

    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if not output.exists():
        return False

    if output.stat().st_size < 10000:
        output.unlink(missing_ok=True)
        return False

    return True


# =========================
# BUILD WORD INDEX
# =========================

word_index = defaultdict(list)

for align in ALIGN_DIR.glob("*.align"):

    with open(align) as f:
        for line in f:

            parts = line.strip().split()
            if len(parts) != 3:
                continue

            _, _, word = parts

            if word in IGNORE:
                continue

            word_index[word].append(align)


# =========================
# MAIN
# =========================

for word in tqdm(VOCAB):

    files = word_index.get(word, [])
    if not files:
        continue

    count = 0

    for align_file in files:

        if count >= CLIPS_PER_WORD:
            break

        video_file = find_video(align_file.stem)
        if video_file is None:
            continue

        with open(align_file) as f:
            for line in f:

                parts = line.strip().split()
                if len(parts) != 3:
                    continue

                start, end, w = parts

                if w != word:
                    continue

                start = int(start) / 10000
                end = int(end) / 10000

                if (end - start) < 0.15:
                    continue

                clip_start, duration, viseme = best_window(
                    video_file, start, end
                )

                viseme_dir = OUTPUT_DIR / viseme
                viseme_dir.mkdir(parents=True, exist_ok=True)

                output = viseme_dir / f"{word}_{count+1:02d}.mp4"

                if save_clip(video_file, clip_start, duration, output):
                    count += 1

                break

    print("Done:", word)

print("\nDataset creation finished.")

  2%|▏         | 1/51 [00:04<03:22,  4.06s/it]

Done: a


  4%|▍         | 2/51 [00:22<10:18, 12.63s/it]

Done: again


  6%|▌         | 3/51 [00:42<12:41, 15.86s/it]

...

Done: y


 98%|█████████▊| 50/51 [13:59<00:11, 11.10s/it]

Done: z


100%|██████████| 51/51 [14:07<00:00, 16.62s/it]

Done: zero

Dataset creation finished.
